# FUnction Calling

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)

ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
API_KEY    = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")


BASE_URL   = f"{ENDPOINT}/openai/v1/"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [2]:
import requests,json
def get_current_weather(city:str)->dict:
    """ can be used to get/fetch current weather information for a city name
    """
    api_key = "6a8b0ac166a37e2b7a38e64416b3c3fe"

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = response.content.decode()
    response = json.loads(response)
    output = {"City Name":city,"weather":response["weather"][0]['description'],
              "temperature":response['main']['temp'],
              "unit":"kelvin"}

    return output

In [4]:
get_current_weather("Pune")

{'City Name': 'Pune',
 'weather': 'scattered clouds',
 'temperature': 306.09,
 'unit': 'kelvin'}

In [10]:
tools = [{"type":"function",
        "name":"get_current_weather",
        "description":"this function can be used to get the current weather information for any given city.",
        "parameters":{
            "type":"object",
            "properties":{"city":{"type":"string","description":"name of city e.g. mumbai, new york"}},
        },
        "required":['city'],
    },
    
]


In [14]:
response = client.responses.create(model=DEPLOYMENT,
                                  input='What is AI?',
                                  tools=tools)
print(response.to_json(indent=2))

{
  "id": "resp_03095a751cfaee4d00699c32f7fb308197908665dafb833b52",
  "created_at": 1771844343.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_03095a751cfaee4d00699c32f838fc81979fd03d5c89248298",
      "content": [
        {
          "annotations": [],
          "text": "Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think and learn. It encompasses a variety of technologies and methods, allowing machines to perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, solving problems, making decisions, and learning from experience.\n\n### Key Aspects of AI:\n\n1. **Machine Learning (ML)**: A subset of AI where algorithms improve through experience. It enables systems to learn from data and make predictions.\n\n2. **Natural Language Processing (

In [13]:

response = client.responses.create(model=DEPLOYMENT,
                                  input='What is the weather in Delhi today?',
                                  tools=tools)
print(response.to_json(indent=2))

{
  "id": "resp_014c18923ee6569400699c32cce918819098551d18bb1ee184",
  "created_at": 1771844300.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini",
  "object": "response",
  "output": [
    {
      "arguments": "{\"city\":\"Delhi\"}",
      "call_id": "call_8i02QGB69zx0Pz7ji8qt0OzS",
      "name": "get_current_weather",
      "type": "function_call",
      "id": "fc_014c18923ee6569400699c32cd6804819086265358214280d6",
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_current_weather",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "name of city e.g. mumbai, new york"
          }
        },
        "additionalProperties": false,
        "required": [
          "city"
        ]
      },
      "strict": true,
      "type

In [24]:
import json

In [25]:
tool_map = {"get_current_weather":get_current_weather}



def generate(prompt):

    input = [{"role":"user","content":prompt}]

    first_response = client.responses.create(model=DEPLOYMENT,
                                            input=input,
                                            instructions='you are a helpful assistant',
                                            tools=tools)
    
    first_output = first_response.output
    function_call = [item for item in first_output if item.type=="function_call"]

    if function_call:
        print("LLM decided to trigger a function call",first_output)
        input+=first_output

        for tool_call in first_output:
            tool_name = tool_call.name
            tool_args = json.loads(tool_call.arguments)
            tool_func = tool_map[tool_name]
            tool_output = tool_func(**tool_args)
            input += [{"type":"function_call_output",
                      "call_id":tool_call.call_id,"output":json.dumps(tool_output),
                      }]
        

        second_response = client.responses.create(model=DEPLOYMENT,
                                                  input=input)
        
        return second_response.output_text
            



        # do something
    else:
        return first_response.output_text


In [22]:
generate("What is AI?")

'Artificial Intelligence (AI) refers to the simulation of human intelligence in machines. It encompasses a range of technologies that enable computers to perform tasks that typically require human intelligence. These tasks include:\n\n1. **Learning**: The ability to improve performance based on experience (machine learning).\n2. **Reasoning**: Making decisions or solving problems based on available information.\n3. **Understanding Natural Language**: Analyzing and interpreting human language (natural language processing).\n4. **Perception**: Interpreting sensory data (like vision or sound).\n5. **Robotics**: Designing machines that can perform physical tasks.\n\nAI can be categorized into two main types:\n\n1. **Narrow AI**: Systems designed to handle specific tasks, such as voice assistants and recommendation algorithms.\n2. **General AI**: A theoretical form of AI that possesses the ability to understand, learn, and apply intelligence broadly like a human.\n\nAI technologies are incr

In [26]:
generate("What is the current weather in mumbai?")

LLM decided to trigger a function call [ResponseFunctionToolCall(arguments='{"city":"mumbai"}', call_id='call_hshDI2XMiHjiRVwFbeNZ3OU9', name='get_current_weather', type='function_call', id='fc_0c971f988b93369a00699c36a0643c8195a60faf7b7062af36', status='completed')]


'The current weather in Mumbai is clear sky, with a temperature of approximately 300.49 K (about 27.3°C or 81.1°F).'

In [28]:
print(generate("What is the current weather in mumbai and pune?"))

LLM decided to trigger a function call [ResponseFunctionToolCall(arguments='{"city":"mumbai"}', call_id='call_fVfknAPfXwPJSKvjciqRnAAK', name='get_current_weather', type='function_call', id='fc_0b611069f2d5965c00699c36f2821c81939701582b344f85bc', status='completed'), ResponseFunctionToolCall(arguments='{"city":"pune"}', call_id='call_hhPFt1ABtULLu2RRtbb0PCrF', name='get_current_weather', type='function_call', id='fc_0b611069f2d5965c00699c36f2b1308193a52ce382d1d0a871', status='completed')]
The current weather in Mumbai is clear sky with a temperature of approximately 27.34°C (300.49 K). 

In Pune, the weather is partly cloudy with a temperature of around 33.94°C (306.09 K).


In [29]:
import wikipedia

def get_wikipedia_summary(query:str):
    response = wikipedia.summary(query)
    return response

In [30]:
get_wikipedia_summary("Second world war")

"World War II or the Second World War (1 September 1939 – 2 September 1945) was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated. Tanks and aircraft played major roles, the latter enabling the strategic bombing of cities and delivery of the only nuclear weapons used in war. World War II is the deadliest conflict in history, causing the death of over 60 million people. Millions died in genocides, including the Holocaust, and by massacres, starvation, and disease. After the Allied victory, Germany, Austria, Japan, and Korea were occupied, and German and Japanese leaders were tried for war crimes.\nThe causes of World War II included unresolved tensions in the aftermath of World War I, and the rise of fascism in Europe and militarism in Japan. Key events preceding the war included Japan's invasion of Manchuria in 1931, the Spanish Civil War, the outbreak of the Second Sino-Japanese War in 1937, and Germany's annexat

In [31]:
tools = [{"type":"function",
        "name":"get_current_weather",
        "description":"this function can be used to get the current weather information for any given city.",
        "parameters":{
            "type":"object",
            "properties":{"city":{"type":"string","description":"name of city e.g. mumbai, new york"}},
        },
        "required":['city'],
    },
    {"type":"function",
        "name":"get_wikipedia_summary",
        "description":"this function can be used to get summary of events, places and people from wikipedia, to be used for specific nouns/poeple/places or events only ",
        "parameters":{
            "type":"object",
            "properties":{"query":{"type":"string","description":"name of place, location, people and event e.g. World War II, Mumbai"}},
        },
        "required":['query'],
    },
    
]


In [32]:
tool_map = {"get_current_weather":get_current_weather,
            "get_wikipedia_summary":get_wikipedia_summary}



def generate(prompt):
    input = [{"role":"user","content":prompt}]
    while True:
        response = client.responses.create(model=DEPLOYMENT, input=input,
                                                instructions='you are a helpful assistant',
                                                tools=tools)
        
        output = response.output
        function_call = [item for item in output if item.type=="function_call"]
        if function_call:
            print("LLM decided to trigger a function call",output)
            input+=output

            for tool_call in output:
                tool_name = tool_call.name
                tool_args = json.loads(tool_call.arguments)
                tool_func = tool_map[tool_name]
                tool_output = tool_func(**tool_args)
                input += [{"type":"function_call_output",
                        "call_id":tool_call.call_id,"output":json.dumps(tool_output),
                        }]
           

        else:
            break
    return response.output_text


In [33]:
generate("what is AI?")

'Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think and learn. It encompasses various technologies and approaches that enable computers to perform tasks that typically require human-like cognitive functions, such as:\n\n1. **Learning**: Acquiring information and rules for using it.\n2. **Reasoning**: Using rules to reach approximate or definite conclusions.\n3. **Problem-solving**: Finding solutions to complex issues.\n4. **Perception**: Interpreting sensory data to understand the environment, such as recognizing speech or images.\n5. **Language Understanding**: Processing and understanding human languages, enabling communication.\n\nAI systems can be classified into two main types:\n\n1. **Narrow AI**: Specialized systems designed to perform a specific task (e.g., voice assistants, recommendation systems).\n2. **General AI**: Hypothetical systems that possess the ability to understand, learn, and apply knowledge broadly, similar

In [34]:
generate("what is weather in mumbai today?")

LLM decided to trigger a function call [ResponseFunctionToolCall(arguments='{"city":"mumbai"}', call_id='call_KDnei78KDkp22HKYZcJK4C5L', name='get_current_weather', type='function_call', id='fc_0d977bf303db77c400699c3a4af4dc8196b9223111e9da2214', status='completed')]


"Today's weather in Mumbai is clear skies with a temperature of approximately 300.49 K, which is about 27.34°C."

In [36]:
print(generate("Tell more about history of mumbai and its current weather information."))

LLM decided to trigger a function call [ResponseFunctionToolCall(arguments='{"query":"Mumbai"}', call_id='call_ASjcCFNJmEkCUGV2jTgYOZdT', name='get_wikipedia_summary', type='function_call', id='fc_03257049ed60e94200699c3a9602b881958ce577e2edd7406c', status='completed'), ResponseFunctionToolCall(arguments='{"city":"Mumbai"}', call_id='call_TTlUQ0szaHmfiDGadRD25ypF', name='get_current_weather', type='function_call', id='fc_03257049ed60e94200699c3a9624488195827b4d53dfd6b9b1', status='completed')]
### History of Mumbai

Mumbai, formerly known as Bombay, is the capital city of Maharashtra, India. It is the financial and commercial hub of the country and one of the most populous cities in the world.

- **Early History**: The area has been inhabited for centuries, initially home to Marathi-speaking Koli communities. The seven islands of what is now Mumbai came under various indigenous rulers before being ceded to the Portuguese in the 16th century and later to the East India Company in 1661.


In [40]:
print(generate("tell me more about city where world war II started and its current weather information."))

LLM decided to trigger a function call [ResponseFunctionToolCall(arguments='{"query":"World War II"}', call_id='call_sG7CrnrAvoJkuNV3VNee81Dz', name='get_wikipedia_summary', type='function_call', id='fc_06bc09dd897093c400699c3b0c8bac8195af380dfde21b60b1', status='completed'), ResponseFunctionToolCall(arguments='{"city":"Gdansk"}', call_id='call_bxXMsToQ4Sj01VQN0dyQ7iYw', name='get_current_weather', type='function_call', id='fc_06bc09dd897093c400699c3b0cb0888195a2a10b10f877daa2', status='completed')]
World War II officially began with the invasion of Poland by Germany on September 1, 1939. The city where this inciting event took place is **Gdańsk**, Poland.

### Summary of World War II
World War II was a global conflict involving most of the world's nations divided into two opposing military alliances: the Allies and the Axis. It was marked by significant events such as the Holocaust, the mobilization of economies and populations, and the extensive use of new military technologies. The 